# TriPoDPy Mini-Tutorial

## Installation

- Install a python version *below* 3.14 (there's a simframe bug still to be fixed)
- clone the repository: `git clone https://github.com/tripod-code/tripodpy.git`
- go into the code repository and install it

```bash
cd tripodpy
pip install .
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
import astropy.constants as c
import tripodpy

au = c.au.cgs.value
yr = (1 * u.year).cgs.value

## Default Setup

In [ ]:
sim = tripodpy.Simulation()

The following `initialize` call sets up all the defaults. You can modify simple things in the `sim.ini` object before calling `initialize`. Things changed afterwards might need to call `sim.update()` (or might be overwritten by default updaters, see the doc for details).

In [ ]:
sim.ini.gas.alpha = 1e-4
sim.initialize()

This runs the simulation, should take just a few seconds. The times of the snapshots are stored in `sim.t.snapshots`. Default only goes to $10^5$ years (takes just a few seconds) - you probably want more, so let's adjust that to a million years (takes ~2 minutes).

In [ ]:
my_snapshots = np.geomspace(1e3, 1e6, 30) * yr
sim.t.snapshots = my_snapshots

In [ ]:
sim.writer.overwrite = True
sim.run()

### Plot default results

Plot surface densities

In [ ]:
fig, ax = plt.subplots()
ax.loglog(sim.grid.r / au, sim.gas.Sigma, label='gas')
ax.loglog(sim.grid.r / au, sim.dust.Sigma.sum(-1), label='dust')
ax.set(ylim=[1e-5, 1e3]);

We can also recompute a size distribution from the simulation parameters:

In [ ]:
a, a_i, sig_da = tripodpy.utils.sim_size_distribution(sim)

fig, ax = plt.subplots()
cf = ax.pcolormesh(sim.grid.r / au, a, sig_da.T, norm='log', vmin=1e-5, vmax=1e0)
ax.loglog()
fig.colorbar(cf)
ax.set(xlim=[1, 3e2]);

But there are also pre-made plotting routines:

In [ ]:
tripodpy.plot.panel('data', it=30)

## Setup with substructure

a simple way is to overwrite the surface densities after initialization

In [ ]:
sim2 = tripodpy.Simulation()
sim2.ini.gas.alpha = 1e-4
sim2.initialize()

Make up a multiplicative gap profile

In [ ]:
gap_profile = 1 - 0.8 * np.exp(-(sim2.grid.r - 40 * au)**2 / (2 * (10 * au)**2))

fig, ax = plt.subplots()
ax.loglog(sim2.grid.r / au, gap_profile);

Now overwrite the surface densities. The `update` call makes sure that all derived quantities (e.g. mid-plane densities, dust-to-gas ratios) are updated as well.

In [ ]:
# the gas is always made of one default component. sim.gas.Sigma is just derived from those
sim2.components.Default.gas.Sigma *= gap_profile

# with just one dust component, the dust can be modified directly. 
sim2.dust.Sigma *= gap_profile[:, None]

# now the gap would viscously get smoothed away. To keep it as it is in a steady
# state, we also modify the viscosity parameter accordingly

sim2.gas.alpha /= gap_profile

# use the same snapshots
sim2.t.snapshots = my_snapshots

# now update everything and select another output folder
sim2.update()
sim2.writer.datadir = 'data_gap'
sim2.writer.overwrite = True

In [ ]:
sim2.run()

### Plot substructure results

In [ ]:
tripodpy.plot.panel('data_gap', it=30)

## Compare

In [ ]:
fig, axs = plt.subplots(2, 1, sharex=True, sharey=True, figsize=(6, 6))

axs[0].loglog(sim.grid.r / au, sim.gas.Sigma, 'k-', label='no substructure')
axs[0].loglog(sim2.grid.r / au, sim2.gas.Sigma, 'k--', label='with substructure')

axs[1].loglog(sim.grid.r / au, sim.dust.Sigma.sum(-1), 'k-', label='no substructure')
axs[1].loglog(sim2.grid.r / au, sim2.dust.Sigma.sum(-1), 'k--', label='with substructure')

axs[0].set_ylabel(r'$\Sigma_\mathrm{g}$ [g cm$^{-2}$]')
axs[0].legend()
axs[1].set_ylabel(r'$\Sigma_\mathrm{d}$ [g cm$^{-2}$]')
axs[1].set_xlabel('r [au]')
axs[1].set(ylim=[1e-5, 1e3]);